In [ ]:
import os
import sys

import pandas as pd
import torch

from models.gnn_model import GNNModel
sys.path.append(os.path.abspath(os.path.join('..')))
from torch_geometric.loader import DataLoader
from utils import mol_to_graph
from utils import MLFlowManager


from joblib import Parallel, delayed


parquet_path = "parquets/df_ml_with_scaffold.parquet"
df = pd.read_parquet(parquet_path)

print("SMILES to graph conversion...")
dataset = Parallel(n_jobs=-1)(
    delayed(mol_to_graph)(s, y) for s, y in zip(df['canonical_smiles'], df['pic50'])
)
# dataset = [mol_to_graph(s, y) for s, y in zip(df['canonical_smiles'], df['pic50'])]
dataset = [d for d in dataset if d is not None] 

mf = MLFlowManager(experiment_name="ChEMBL_GNN_Scaffold_Split")

model_types = ["GCN", "GIN"]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for m_type in model_types:

    model = GNNModel(num_node_features=4, hidden_channels=64, model_type=m_type)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    loader = DataLoader(dataset, batch_size=32, shuffle=True)
    
    run_name = f"Run_{m_type}_ScaffoldData"
    print(f"Starting: {run_name}")
    
    model.train_gnn(
        model=model, 
        loader=loader, 
        optimizer=optimizer, 
        device=device, 
        mf_manager=mf, 
        run_name=run_name
    )

Konwersja SMILES na grafy...


[17:34:07] Explicit valence for atom # 17 P, 7, is greater than permitted
[17:41:15] Explicit valence for atom # 19 P, 7, is greater than permitted
[17:45:32] Explicit valence for atom # 1 P, 7, is greater than permitted
